# 02 - Descriptor stability vs. images-per-class (MANDATORY before Phase 1)

AGENTS.md Sec 3.3: if the per-class image quota is too small, class mean and within-class
covariance are noisy, and that INPUT noise depresses Phase-1 R2. A negative Phase-1 result
would then be **ambiguous** - no signal, or a bad descriptor? This fixes the quota at a point
where the descriptor has already stabilized, so the gate becomes interpretable.

**Method:** for each quota q, draw TWO DISJOINT subsets of q images/class, build phi(y) on
each independently, correlate every descriptor feature across classes. Needs >= 2q per class.

**Pre-registered pass criteria:** report the stability curve with SE; `recommended_quota` =
smallest q with mean cross-draw correlation >= 0.90. If no q reaches 0.90, that is itself the
finding (report it; do not silently pick the largest q).

**Grid extended to q=200 (2026-07-25), recorded:** the first run crossed 0.90 only at q=100,
which was also the LARGEST testable q (200/class extracted, disjoint pairs need 2q). The curve
was still rising (0.852 -> 0.903), so the plateau was unknown. Extraction is now 400/class so
q=200 is testable. This EXTENDS the grid; the threshold and the rule are unchanged - it is not
moving the goalposts. See reports/descriptor_stability_findings.md.

No GPU needed - this reads the embeddings 00a already wrote to Drive.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'cifar100'
BACKBONE   = 'resnet50_self'
QUOTAS     = (10, 25, 50, 100, 200)  # needs >= 2*max(QUOTAS) images/class extracted
N_REPS     = 5
STABLE_THRESHOLD = 0.90
SEED = 42
EMB_DIR = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_DIR =', EMB_DIR)


## 2. Mount Drive + repo + env + seed


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load train-subset embeddings (descriptors use TRAINING data only, Sec 6.3)


In [ ]:
import numpy as np, os
path = os.path.join(EMB_DIR, 'train_subset.npz')
assert os.path.exists(path), f'missing {path} - run 00a first'
d = np.load(path)
emb, lab, lg = d['embeddings'], d['labels'], d['logits']
n_classes = int(lab.max()) + 1
counts = np.bincount(lab, minlength=n_classes)
print(f'embeddings={emb.shape} classes={n_classes} per-class min/median/max = '
      f'{counts.min()}/{int(np.median(counts))}/{counts.max()}')
need = 2 * max(QUOTAS)
if counts.min() < need:
    print(f'WARNING: need >= {need} images/class for q={max(QUOTAS)} disjoint pairs; '
          f'min is {counts.min()}. Quotas above {counts.min()//2} will be skipped.')
    print('Fix: re-run 00a cell 6 with PER_CLASS_TRAIN_QUOTA =', need)


## 4. Run stability study


In [ ]:
from pcc.descriptors.stability import descriptor_stability
res = descriptor_stability(emb, lg, lab, n_classes, quotas=QUOTAS,
                           n_reps=N_REPS, seed=SEED, stable_threshold=STABLE_THRESHOLD)
print(f"{'quota':>6} {'mean_corr':>10} {'se':>8} {'classes':>8}")
for q in QUOTAS:
    v = res['by_quota'].get(q, {})
    if v.get('insufficient_data'): print(f'{q:>6} {"skipped (too few images/class)":>28}'); continue
    print(f"{q:>6} {v['mean_corr']:>10.3f} {v['se']:>8.3f} {v['n_classes_used']:>8}")
print('\nrecommended_quota:', res['recommended_quota'], f'(threshold {STABLE_THRESHOLD})')


## 5. Per-feature detail (which descriptors are the noisy ones)


In [ ]:
names = res['feature_names']
avail = [q for q in QUOTAS if not res['by_quota'].get(q,{}).get('insufficient_data')]
print('feature'.ljust(20) + ''.join(f'q={q}'.rjust(9) for q in avail))
for n in names:
    row = ''.join(f"{res['by_quota'][q]['per_feature'][n]:9.3f}" for q in avail)
    print(n.ljust(20) + row)
print('\nexcluded from aggregate (constant by construction):',
      res['by_quota'][avail[0]]['excluded_from_aggregate'])


## 6. Write report + quota marker


In [ ]:
import time, json, os
from pcc.utils.io import write_report

rec = res['recommended_quota']
conclusion = (f'PASS - descriptor stable at quota {rec}' if rec is not None
              else f'NO QUOTA REACHED {STABLE_THRESHOLD} - report as finding; Phase-1 negatives '
                   f'would be ambiguous at these quotas')
clean = {'by_quota': {str(k): v for k, v in res['by_quota'].items()},
         'feature_names': res['feature_names'],
         'recommended_quota': rec, 'stable_threshold': STABLE_THRESHOLD}
report = write_report('pcc/reports', f'02_descriptor_stability_{DATASET}',
    hypothesis='class descriptors phi(y) stabilize as images-per-class grows',
    pass_criteria=f'report stability curve with SE for q in {QUOTAS}; recommended_quota = '
                  f'smallest q with mean cross-draw correlation >= {STABLE_THRESHOLD}; '
                  f'if none reaches it, report that as the finding',
    config=dict(dataset=DATASET, backbone=BACKBONE, quotas=list(QUOTAS), n_reps=N_REPS),
    seed=SEED, results=clean, conclusion=conclusion, started_at=time.time())
print('report:', report)

if rec is not None:
    mark = f'{DRIVE_ROOT}/gates/DESCRIPTOR_QUOTA_{DATASET}.json'
    os.makedirs(os.path.dirname(mark), exist_ok=True)
    json.dump({'dataset':DATASET,'recommended_quota':rec,'threshold':STABLE_THRESHOLD},
              open(mark,'w'), indent=2)
    print('quota marker written ->', mark, '| Phase 1 may proceed with quota', rec)
else:
    print('NO marker written - decide with a human before running Phase 1 (Sec 3.3).')
